# Config and setup

### Before we begin

- `uv sync` (or `pip install amoorion` and the packages of your platforms)
- `cp example.env .env`: this file only says where the vault is. Secrets never go
  in it, nor in the repository.
- `uv run task vault:new` when you use KeePassXC: it makes the file and an empty
  entry per platform.

### Basic, simple

With no arguments it takes `.env` of the current directory, as the CLI does.

In [ ]:
import os, tempfile
from pathlib import Path

# Examples stay off your real setup: secrets from environment variables
# (the `env` vault), an inbox and a dataset in a temporary directory.
scratch = Path(tempfile.mkdtemp())
os.environ["AMOORION_VAULT"] = "env"
os.environ["AMOORION_INBOX"] = str(scratch / "inbox.sqlite3")
os.environ["AMOORION_DATASET"] = str(scratch / "dataset")

In [ ]:
from amoorion import Amoorion

amo = Amoorion()
amo

### Three variants of config, three vaults

An env file belongs to one object and doesn't touch the environment of the
process. Opening the vault waits for its first use, so these three can be looked
at without any server.

In [ ]:
variants = {
    "keepassxc": "AMOORION_VAULT=keepassxc\nAMOORION_VAULT_PATH=./dataset/.secrets.kdbx\n",
    "bitwarden": "AMOORION_VAULT=bitwarden\nAMOORION_VAULT_ADDR=https://vault.example.org\n",
    "openbao": "AMOORION_VAULT=http\nAMOORION_VAULT_ADDR=https://bao.example.org\nAMOORION_VAULT_PATH=secret/amoorion\n",
}
for label, text in variants.items():
    (scratch / f"{label}.env").write_text(text)

amo1 = Amoorion(scratch / "keepassxc.env")
amo2 = Amoorion(scratch / "bitwarden.env", scratch / "dataset-two")
amo3 = Amoorion(scratch / "openbao.env", scratch / "dataset-three", flags={"mode": "mcp"})
for one in (amo1, amo2, amo3):
    print(one)

### Flags

`mode` says which front-end `amo.run()` starts: `cli` (default), `api` or `mcp`.

In [ ]:
cli = Amoorion(flags={"mode": "cli"})
cli.run("--json", "inbox") == 0

### States

`save` writes where things are, and hashes to detect changes; `load` takes it back.
No secret is written: of the vault only its kind, address and path are hashed.

In [ ]:
cli.use("dryrun")
state = scratch / "state.yaml"
cli.save(state)
print(state.read_text())

In [ ]:
restored = Amoorion().load(state)
print("changed so far:", restored.changes())
restored.publish(text="something happens")
restored.read(limit=1)
print("changed now:", restored.changes())
print("after saving:", restored.load(state).save(state).changes())